In [ ]:
#| default_exp roofline

In [ ]:
#| include: false
from nbdev.showdoc import *

In [ ]:
#| export
from __future__ import annotations

import math
import time
import warnings
from dataclasses import dataclass, asdict
from contextlib import contextmanager

import numpy as np
import torch
import torch.nn as nn
import plotly.graph_objects as go

from fasterbench.core import _device_ctx, _on_device, _sync, _fmt_float, _section
from fasterbench.profiling import _leaf_modules, _tensor_bytes, _output_bytes

In [ ]:
#| export
@dataclass(slots=True)
class HardwarePeaks:
    "Empirically measured achievable peak compute and streaming bandwidth for a device."
    peak_flops: float       # achievable peak FLOPs/s
    peak_bandwidth: float   # achievable streaming bandwidth in bytes/s
    ridge_point: float      # FLOPs/byte = peak_flops / peak_bandwidth
    device: str             # e.g. "cuda:0", "cpu"
    dtype: str              # e.g. "torch.float32"
    tf32_enabled: bool      # whether matmul TF32 was on during probe
    cudnn_benchmark: bool   # whether cudnn.benchmark was on during probe

    def as_dict(self) -> dict:
        return asdict(self)

In [ ]:
show_doc(HardwarePeaks)

In [ ]:
#| export
@contextmanager
def _pinned_benchmark_flags(tf32: bool = False):
    "Pin matmul TF32 and cudnn.benchmark during probe; restore on exit."
    if torch.cuda.is_available():
        prev_tf32 = torch.backends.cuda.matmul.allow_tf32
        prev_cudnn_tf32 = torch.backends.cudnn.allow_tf32
        prev_bench = torch.backends.cudnn.benchmark
        torch.backends.cuda.matmul.allow_tf32 = tf32
        torch.backends.cudnn.allow_tf32 = tf32
        torch.backends.cudnn.benchmark = False
        try:
            yield (tf32, False)
        finally:
            torch.backends.cuda.matmul.allow_tf32 = prev_tf32
            torch.backends.cudnn.allow_tf32 = prev_cudnn_tf32
            torch.backends.cudnn.benchmark = prev_bench
    else:
        yield (False, False)

In [ ]:
#| export
_PEAKS_CACHE: dict = {}


def measure_peaks(
    device: str | torch.device = "cuda",  # device to probe
    *,
    dtype: torch.dtype = torch.float32,   # probe precision
    matmul_size: int = 4096,              # N for NxN matmul probe
    bandwidth_mb: int = 256,              # per-buffer size in MiB (auto-bumped above L3)
    warmup: int = 5,                      # warmup iterations
    steps: int = 20,                      # measurement iterations (report max)
    allow_tf32: bool = False,             # pin TF32 off by default for honest fp32 peak
    cache: bool = True,                   # cache per (device, dtype, sizes)
) -> HardwarePeaks:
    "Empirically probe achievable peak FLOPs/s and streaming bandwidth."
    dev = torch.device(device) if isinstance(device, str) else device
    if dev.type == "cuda" and not torch.cuda.is_available():
        warnings.warn("CUDA requested but not available - falling back to CPU")
        dev = torch.device("cpu")
    key = (str(dev), str(dtype), matmul_size, bandwidth_mb, allow_tf32)
    if cache and key in _PEAKS_CACHE:
        return _PEAKS_CACHE[key]

    with _pinned_benchmark_flags(tf32=allow_tf32) as (tf32_on, bench_on):
        # --- peak FLOPs probe ---
        N = matmul_size
        a = torch.randn(N, N, device=dev, dtype=dtype)
        b = torch.randn(N, N, device=dev, dtype=dtype)
        for _ in range(warmup):
            (a @ b)
            _sync(dev)
        flops_per_matmul = 2.0 * N * N * N
        best_flops = 0.0
        c = None
        for _ in range(steps):
            _sync(dev)
            t0 = time.perf_counter()
            c = a @ b
            _sync(dev)
            dt = time.perf_counter() - t0
            if dt > 0:
                best_flops = max(best_flops, flops_per_matmul / dt)
        del a, b, c
        if dev.type == "cuda":
            torch.cuda.empty_cache()

        # --- bandwidth probe (cache-defeating) ---
        # target buffer sized to blow past L3 and any GPU L2.
        buf_bytes = max(bandwidth_mb * 1024 * 1024, 64 * 1024 * 1024)
        n_elems = buf_bytes // dtype.itemsize
        src = torch.empty(n_elems, device=dev, dtype=dtype).normal_()
        dst = torch.empty(n_elems, device=dev, dtype=dtype).normal_()
        for _ in range(warmup):
            dst.copy_(src)
            _sync(dev)
        bytes_moved_per_copy = 2.0 * n_elems * dtype.itemsize  # read + write
        best_bw = 0.0
        for i in range(steps):
            # alternate direction to defeat any residency assumptions
            s, d = (src, dst) if (i % 2 == 0) else (dst, src)
            _sync(dev)
            t0 = time.perf_counter()
            d.copy_(s)
            _sync(dev)
            dt = time.perf_counter() - t0
            if dt > 0:
                best_bw = max(best_bw, bytes_moved_per_copy / dt)
        del src, dst
        if dev.type == "cuda":
            torch.cuda.empty_cache()

        result = HardwarePeaks(
            peak_flops=best_flops,
            peak_bandwidth=best_bw,
            ridge_point=best_flops / best_bw if best_bw > 0 else float("inf"),
            device=str(dev),
            dtype=str(dtype),
            tf32_enabled=tf32_on,
            cudnn_benchmark=bench_on,
        )
    if cache:
        _PEAKS_CACHE[key] = result
    return result


def clear_peaks_cache() -> None:
    "Reset the measure_peaks() cache."
    _PEAKS_CACHE.clear()

In [ ]:
show_doc(measure_peaks)

In [ ]:
#| export
@dataclass(slots=True)
class RooflinePoint:
    "Per-layer roofline measurement. Bytes formula: weight_bytes + input_bytes + output_bytes (each counted once per forward call, per Williams 2009)."
    name: str
    type: str
    flops: float                  # total FLOPs (2 * MACs)
    bytes_moved: float            # weights + input + output bytes per forward call
    time_s: float                 # measured wall time (mean over steps)
    arithmetic_intensity: float   # flops / bytes_moved
    achieved_gflops: float        # flops / time / 1e9
    bound: str                    # "memory" | "compute" | "undefined"
    utilization_pct: float        # achieved / roof * 100

    def as_dict(self) -> dict:
        return asdict(self)

In [ ]:
show_doc(RooflinePoint)

In [ ]:
#| export
def _layer_flops(module: nn.Module, inp, output) -> float:
    "Estimate FLOPs for a single leaf module forward call. Returns 0 for layer types we do not model."
    # Covers the layer types where FLOPs actually dominate - conv, linear, matmul-adjacent.
    # Cheap element-wise ops (ReLU, BN, pooling) are intentionally assigned 0 FLOPs; they end up
    # in the \"undefined\" bucket (memory-bound trivially) and are surfaced via the warning.
    t_in = inp[0] if isinstance(inp, tuple) and len(inp) > 0 and isinstance(inp[0], torch.Tensor) else None
    t_out = output if isinstance(output, torch.Tensor) else None

    if isinstance(module, (nn.Conv1d, nn.Conv2d, nn.Conv3d)) and t_out is not None:
        cin = module.in_channels // module.groups
        cout = module.out_channels
        ksize = 1
        for k in (module.kernel_size if isinstance(module.kernel_size, tuple) else (module.kernel_size,)):
            ksize *= k
        spatial = 1
        for s in t_out.shape[2:]:
            spatial *= int(s)
        batch = int(t_out.shape[0]) if t_out.ndim > 0 else 1
        macs = batch * cin * cout * ksize * spatial
        flops = 2.0 * macs
        if module.bias is not None:
            flops += batch * cout * spatial  # bias add
        return float(flops)

    if isinstance(module, (nn.ConvTranspose1d, nn.ConvTranspose2d, nn.ConvTranspose3d)) and t_in is not None:
        cin = module.in_channels // module.groups
        cout = module.out_channels
        ksize = 1
        for k in (module.kernel_size if isinstance(module.kernel_size, tuple) else (module.kernel_size,)):
            ksize *= k
        spatial = 1
        for s in t_in.shape[2:]:
            spatial *= int(s)
        batch = int(t_in.shape[0]) if t_in.ndim > 0 else 1
        return float(2.0 * batch * cin * cout * ksize * spatial)

    if isinstance(module, nn.Linear) and t_in is not None:
        in_features = module.in_features
        out_features = module.out_features
        # Everything before the last dim is treated as batch.
        batch = 1
        for s in t_in.shape[:-1]:
            batch *= int(s)
        flops = 2.0 * batch * in_features * out_features
        if module.bias is not None:
            flops += batch * out_features
        return float(flops)

    return 0.0


#| export
def _setup_roofline_hooks(
    leaf_modules,     # {name: module} for leaf modules
    bytes_state,      # {name: []} to accumulate bytes per call
    time_state,       # {name: []} to accumulate seconds per call
    flops_state,      # {name: []} to accumulate FLOPs per call
    call_state,       # {name: int} counter of forward calls
    device_type,      # "cuda" or "cpu"
):
    "Register hooks to measure (bytes moved, time, FLOPs, call count) per layer."
    handles = []
    is_cuda = device_type == "cuda"

    def make_hooks(name: str, module: nn.Module):
        w_bytes = sum(_tensor_bytes(p) for p in module.parameters(recurse=False))
        if is_cuda:
            state = {"start": None, "end": None}
            def pre(mod, inp):
                state["start"] = torch.cuda.Event(enable_timing=True)
                state["end"] = torch.cuda.Event(enable_timing=True)
                state["start"].record()
                in_b = sum(_tensor_bytes(t) for t in inp if isinstance(t, torch.Tensor))
                bytes_state[name].append(in_b + w_bytes)
            def post(mod, inp, output):
                state["end"].record()
                out_b = _output_bytes(output)
                bytes_state[name][-1] += out_b
                torch.cuda.synchronize()
                time_state[name].append(state["start"].elapsed_time(state["end"]) / 1000.0)
                flops_state[name].append(_layer_flops(mod, inp, output))
                call_state[name] += 1
        else:
            state = {"t0": 0.0}
            def pre(mod, inp):
                state["t0"] = time.perf_counter()
                in_b = sum(_tensor_bytes(t) for t in inp if isinstance(t, torch.Tensor))
                bytes_state[name].append(in_b + w_bytes)
            def post(mod, inp, output):
                dt = time.perf_counter() - state["t0"]
                out_b = _output_bytes(output)
                bytes_state[name][-1] += out_b
                time_state[name].append(dt)
                flops_state[name].append(_layer_flops(mod, inp, output))
                call_state[name] += 1
        return pre, post

    for name, mod in leaf_modules.items():
        pre, post = make_hooks(name, mod)
        handles.append(mod.register_forward_pre_hook(pre))
        handles.append(mod.register_forward_hook(post))
    return handles

In [ ]:
#| export
class RooflineAnalyzer:
    "Per-layer roofline analysis: measure arithmetic intensity and achieved GFLOPs/s against hardware peaks."

    def __init__(
        self,
        model: nn.Module,                    # model to analyze
        sample: torch.Tensor,                # input tensor (with batch dimension)
        peaks: HardwarePeaks | None = None,  # optional precomputed hardware peaks
    ):
        self.model = model
        self.sample = sample
        self.peaks = peaks
        self._results: list[RooflinePoint] = []

    @property
    def results(self) -> list[RooflinePoint]:
        "Per-layer roofline measurements (populated after profile())."
        return self._results

    @torch.no_grad()
    def profile(
        self,
        *,
        device: str | torch.device = "cuda",  # device to profile on
        warmup: int = 5,                      # warmup iterations
        steps: int = 20,                      # measurement iterations
    ) -> list[RooflinePoint]:
        "Run per-layer profiling: collect FLOPs, bytes, and time, then classify against peaks."
        with _device_ctx(device) as dev:
            if dev.type == "cuda":
                torch.cuda.empty_cache()
            _sync(dev)

            # Ensure peaks are available
            if self.peaks is None:
                self.peaks = measure_peaks(dev, dtype=self.sample.dtype)

            with _on_device(self.model.eval(), dev) as model:
                sample = self.sample.to(dev)
                leaf_mods = _leaf_modules(model)

                bytes_state: dict[str, list] = {n: [] for n in leaf_mods}
                time_state: dict[str, list] = {n: [] for n in leaf_mods}
                flops_state: dict[str, list] = {n: [] for n in leaf_mods}
                call_state: dict[str, int] = {n: 0 for n in leaf_mods}

                handles = _setup_roofline_hooks(
                    leaf_mods, bytes_state, time_state, flops_state, call_state, dev.type,
                )
                try:
                    for _ in range(warmup):
                        model(sample)
                    # Reset accumulators after warmup
                    for n in leaf_mods:
                        bytes_state[n].clear()
                        time_state[n].clear()
                        flops_state[n].clear()
                        call_state[n] = 0
                    for _ in range(steps):
                        model(sample)
                finally:
                    for h in handles:
                        h.remove()

            # Warn if any module was invoked >1x per forward pass (shared module)
            multi_call = [n for n, c in call_state.items() if steps > 0 and c > steps]
            if multi_call:
                warnings.warn(
                    f"{len(multi_call)} module(s) were called more than once per forward "
                    f"pass; their bytes/time are summed across calls. First: {multi_call[:3]}"
                )

        # --- Build RooflinePoint per layer ---
        peak_flops = self.peaks.peak_flops
        peak_bw = self.peaks.peak_bandwidth
        ridge = self.peaks.ridge_point

        results: list[RooflinePoint] = []
        for name, mod in leaf_mods.items():
            b_list = bytes_state[name]
            t_list = time_state[name]
            f_list = flops_state[name]
            bytes_moved = float(np.mean(b_list)) if b_list else 0.0
            time_s = float(np.mean(t_list)) if t_list else 0.0
            flops = float(np.mean(f_list)) if f_list else 0.0

            if flops == 0 or bytes_moved == 0 or time_s == 0:
                results.append(RooflinePoint(
                    name=name, type=mod.__class__.__name__,
                    flops=flops, bytes_moved=bytes_moved, time_s=time_s,
                    arithmetic_intensity=0.0, achieved_gflops=0.0,
                    bound="undefined", utilization_pct=0.0,
                ))
                continue

            ai = flops / bytes_moved
            achieved_gflops = flops / time_s / 1e9
            roof_flops = min(peak_flops, ai * peak_bw)
            roof_gflops = roof_flops / 1e9
            bound = "memory" if ai < ridge else "compute"
            util = (achieved_gflops / roof_gflops * 100) if roof_gflops > 0 else 0.0
            results.append(RooflinePoint(
                name=name, type=mod.__class__.__name__,
                flops=flops, bytes_moved=bytes_moved, time_s=time_s,
                arithmetic_intensity=ai, achieved_gflops=achieved_gflops,
                bound=bound, utilization_pct=util,
            ))

        # Warn if any layer had zero flops/bytes/time -> bound is "undefined"
        zero_flops = [r.name for r in results if r.bound == "undefined"]
        if zero_flops:
            warnings.warn(
                f"{len(zero_flops)} layer(s) have undefined roofline (zero FLOPs, bytes, "
                f"or time). First: {zero_flops[:3]}"
            )

        self._results = results
        return results

    def summary(self, *, top: int = 10) -> None:
        "Print a table of the slowest layers with their roofline metrics."
        if not self._results:
            raise RuntimeError("No results available. Call profile() first.")
        print(_section("Roofline", 72))
        header = f"  {'name':32} {'type':14} {'FLOPs':>10} {'bytes':>10} {'AI':>8} {'GFLOPs/s':>10} {'bound':>9} {'util%':>7}"
        print(header)
        # Sort by measured time, descending (slowest first)
        sorted_rows = sorted(self._results, key=lambda r: r.time_s, reverse=True)[:top]
        for r in sorted_rows:
            flops_str = f"{r.flops/1e6:>8.2f}M" if r.flops >= 1e6 else f"{r.flops:>10.0f}"
            bytes_str = f"{r.bytes_moved/1e6:>8.2f}M" if r.bytes_moved >= 1e6 else f"{r.bytes_moved:>10.0f}"
            ai_str = _fmt_float(r.arithmetic_intensity, width=8, decimals=2)
            gf_str = _fmt_float(r.achieved_gflops, width=10, decimals=2)
            util_str = _fmt_float(r.utilization_pct, width=6, decimals=1) + "%"
            print(f"  {r.name:32} {r.type:14} {flops_str} {bytes_str} {ai_str} {gf_str} {r.bound:>9} {util_str}")

    def plot(
        self,
        *,
        title: str = "Roofline",  # figure title
    ) -> go.Figure:
        "Render the roofline plot with per-layer scatter points on a log-log grid. Markers are colored by bound classification."
        if not self._results:
            raise RuntimeError("No results available. Call profile() first.")
        if self.peaks is None:
            raise RuntimeError("No hardware peaks available.")

        peak_flops = self.peaks.peak_flops
        peak_bw = self.peaks.peak_bandwidth
        ridge = self.peaks.ridge_point
        peak_gflops = peak_flops / 1e9

        # Build the roof curve: y = min(peak_flops, AI * peak_bw) / 1e9
        valid = [r for r in self._results if r.bound != "undefined"]
        if valid:
            ai_min = max(min(r.arithmetic_intensity for r in valid) / 10.0, 1e-3)
            ai_max = max(r.arithmetic_intensity for r in valid) * 10.0
        else:
            ai_min, ai_max = 1e-2, 1e3
        ai_max = max(ai_max, ridge * 10.0)

        ai_grid = np.logspace(math.log10(ai_min), math.log10(ai_max), 200)
        roof_gflops = np.minimum(peak_gflops, ai_grid * peak_bw / 1e9)

        fig = go.Figure()
        fig.add_trace(go.Scatter(
            x=ai_grid, y=roof_gflops, mode="lines",
            line=dict(color="#008080", width=2),
            name=f"Roof ({peak_gflops:.0f} GFLOPs/s, {peak_bw/1e9:.1f} GB/s)",
            hoverinfo="skip",
        ))
        # Ridge point marker
        fig.add_trace(go.Scatter(
            x=[ridge], y=[peak_gflops], mode="markers",
            marker=dict(color="#008080", size=10, symbol="diamond"),
            name=f"Ridge point ({ridge:.1f} FLOP/byte)",
            hovertemplate="Ridge: %{x:.2f} FLOP/byte<extra></extra>",
        ))

        # Color palette for layers
        color_map = {"memory": "#89d6c9", "compute": "#008080"}
        for bound_label in ("memory", "compute"):
            pts = [r for r in valid if r.bound == bound_label]
            if not pts:
                continue
            hover = [
                f"{r.name}<br>{r.type}<br>AI: {r.arithmetic_intensity:.3f} FLOP/byte"
                f"<br>{r.achieved_gflops:.2f} GFLOPs/s<br>util: {r.utilization_pct:.1f}%"
                for r in pts
            ]
            fig.add_trace(go.Scatter(
                x=[r.arithmetic_intensity for r in pts],
                y=[r.achieved_gflops for r in pts],
                mode="markers",
                marker=dict(color=color_map[bound_label], size=8, opacity=0.8,
                            line=dict(color="#008080", width=0.5)),
                name=f"{bound_label}-bound",
                text=hover,
                hovertemplate="%{text}<extra></extra>",
            ))

        fig.update_layout(
            title=title,
            xaxis=dict(title="Arithmetic intensity (FLOP/byte)", type="log"),
            yaxis=dict(title="Achieved performance (GFLOPs/s)", type="log"),
            paper_bgcolor="rgba(0,0,0,0)",
            plot_bgcolor="rgba(0,0,0,0)",
            legend=dict(bgcolor="rgba(0,0,0,0)"),
        )
        return fig

In [ ]:
show_doc(RooflineAnalyzer)

In [ ]:
show_doc(RooflineAnalyzer.profile)

In [ ]:
show_doc(RooflineAnalyzer.summary)

In [ ]:
show_doc(RooflineAnalyzer.plot)

## Usage

```python
from fasterbench.roofline import RooflineAnalyzer

ra = RooflineAnalyzer(model, sample)
ra.profile(device="cuda")
ra.summary()
fig = ra.plot()
fig.show()
```

This is a measurement primitive: it measures, it never prescribes. The decisions that consume `ra.results` live in the compression workflow that calls it.

In [ ]:
#| hide
from fastcore.test import *

_p = measure_peaks("cpu", steps=3, warmup=1, matmul_size=256, bandwidth_mb=32, cache=False)
assert isinstance(_p, HardwarePeaks)
assert _p.peak_flops > 0
assert _p.peak_bandwidth > 0
test_close(_p.ridge_point, _p.peak_flops / _p.peak_bandwidth, eps=1e-6)
assert _p.device == "cpu"
assert _p.tf32_enabled is False
assert _p.cudnn_benchmark is False

In [ ]:
#| hide
clear_peaks_cache()
_p1 = measure_peaks("cpu", steps=2, warmup=1, matmul_size=128, bandwidth_mb=16, cache=True)
_p2 = measure_peaks("cpu", steps=2, warmup=1, matmul_size=128, bandwidth_mb=16, cache=True)
assert _p1 is _p2  # cache hit returns same object

clear_peaks_cache()
_p3 = measure_peaks("cpu", steps=2, warmup=1, matmul_size=128, bandwidth_mb=16, cache=True)
assert _p3 is not _p1  # cache was cleared

In [ ]:
#| hide
# Hand-computed Conv2d test: wrap in Sequential so leaf name is non-empty
import torch
import torch.nn as nn

_synth_peaks = HardwarePeaks(
    peak_flops=1e12, peak_bandwidth=1e11, ridge_point=10.0,
    device="cpu", dtype="torch.float32",
    tf32_enabled=False, cudnn_benchmark=False,
)
_conv = nn.Conv2d(4, 8, kernel_size=3, padding=1, bias=False)
_x = torch.randn(1, 4, 8, 8)
_model = nn.Sequential(_conv)
_ra = RooflineAnalyzer(_model, _x, peaks=_synth_peaks)
_res = _ra.profile(device="cpu", warmup=1, steps=2)
assert len(_res) == 1
_r = _res[0]
# Expected MACs = 4*8*3*3 * 8*8 = 18432; FLOPs = 2*MACs = 36864
test_eq(_r.flops, 36864.0)
# Expected bytes: weights 4*8*3*3*4 = 1152, input 1*4*8*8*4 = 1024, output 1*8*8*8*4 = 2048
# total = 4224
test_eq(_r.bytes_moved, 4224.0)
test_close(_r.arithmetic_intensity, 36864.0 / 4224.0, eps=1e-3)
assert _r.bound in ("memory", "compute")
assert math.isfinite(_r.utilization_pct)

In [ ]:
#| hide
# Tiny Linear stack with synthetic peaks
_m = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 16))
_x = torch.randn(1, 32)
_peaks = HardwarePeaks(
    peak_flops=1e12, peak_bandwidth=1e11, ridge_point=10.0,
    device="cpu", dtype="torch.float32",
    tf32_enabled=False, cudnn_benchmark=False,
)
_ra = RooflineAnalyzer(_m, _x, peaks=_peaks)
_res = _ra.profile(device="cpu", warmup=1, steps=2)
assert len(_res) > 0
for _r in _res:
    assert _r.arithmetic_intensity >= 0
    assert _r.bound in {"memory", "compute", "undefined"}
    assert math.isfinite(_r.utilization_pct)
# profiling leaves the model on the device it came in on
test_eq({p.device.type for p in _m.parameters()}, {"cpu"})
# summary and plot should run without error
_ra.summary(top=5)
_fig = _ra.plot()
assert isinstance(_fig, go.Figure)

In [ ]:
#| hide
#| notest
from torchvision.models import resnet18

_model = resnet18()
_sample = torch.randn(1, 3, 64, 64)
_synth = HardwarePeaks(
    peak_flops=1e13, peak_bandwidth=5e11, ridge_point=20.0,
    device="cpu", dtype="torch.float32",
    tf32_enabled=False, cudnn_benchmark=False,
)
_ra = RooflineAnalyzer(_model, _sample, peaks=_synth)
_results = _ra.profile(device="cpu", warmup=2, steps=3)
assert len(_results) > 0
assert all(r.bound in {"memory", "compute", "undefined"} for r in _results)
_ra.summary(top=5)
_fig = _ra.plot()
assert isinstance(_fig, go.Figure)

In [ ]:
#| hide
#| notest
if torch.cuda.is_available():
    _p = measure_peaks(device="cuda", matmul_size=512, bandwidth_mb=64, steps=3, warmup=1, cache=False)
    assert _p.peak_flops > 0
    assert _p.peak_bandwidth > 0
    assert _p.ridge_point > 0

In [ ]:
#| hide
#| notest
# Needs a second device: profiling a CUDA model on the CPU must not park it there.
if torch.cuda.is_available():
    _cm = nn.Sequential(nn.Linear(32, 64), nn.ReLU(), nn.Linear(64, 16)).cuda()
    _cra = RooflineAnalyzer(_cm, torch.randn(1, 32), peaks=_peaks)
    _cra.profile(device="cpu", warmup=1, steps=2)
    test_eq({p.device.type for p in _cm.parameters()}, {"cuda"})

---

## See Also

- [Per-layer profiling](profiling.html) - Generic per-layer hook infrastructure reused here
- [Compute metrics](../metrics/compute.html) - Model-level FLOPs counting
- [Speed metrics](../metrics/speed.html) - Latency measurement